In [8]:
import torch
import pickle
from tqdm import tqdm
import numpy as np
import random
import pandas as pd

from transformers import AutoTokenizer

from sentence_transformers import models, SentenceTransformer
from sentence_transformers import InputExample
from torch.utils.data import DataLoader
from sentence_transformers.evaluation import TripletEvaluator

import datasets
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator

import matplotlib
%matplotlib inline
from matplotlib import pyplot as plt
from matplotlib import font_manager as fm
from matplotlib import rc

import os
import subprocess



In [1]:
"""
#If any errors occur with the tokenizer, download the tokenizer.
tokenizer = AutoTokenizer.from_pretrained('sentence-transformers/roberta-base-nli-stsb-mean-tokens')

for i in range(5):
    model_path = f"../model/full_data/roberta-base_idx{i}_epoch3/"
    tokenizer.save_pretrained(model_path)"""

'\n#If any errors occur with the tokenizer, download the tokenizer.\ntokenizer = AutoTokenizer.from_pretrained(\'sentence-transformers/roberta-base-nli-stsb-mean-tokens\')\n\nfor i in range(5):\n    model_path = f"../model/full_data/roberta-base_idx{i}_epoch3/"\n    tokenizer.save_pretrained(model_path)'

# Triplet evaluation

* following codes generate 'evaluation' folder and save triplet evaluation results for both fine-tuned and base S-BERT models (Takes few hours)

In [ ]:
""" 
#Triplet evaluation 

# Set GPU
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# Run evaluation (for the finetuned S-BERT model)
subprocess.run([
    "python", "triplet_evaluator_generalized.py",
    "--dataname", "full_data",
    "--trainset_name", "train_triplet", 
    "--testset_name", "test_triplet",
    "--n_data", "5",
    "--num_epochs", "3"
])

# Run evaluation (for the base S-BERT model)
subprocess.run([
    "python", "triplet_evaluator_generalized_beforefinetune.py",
    "--dataname", "full_data",
    "--trainset_name", "train_triplet", 
    "--testset_name", "test_triplet",
    "--n_data", "5",
    "--num_epochs", "3"
])

"""

### 1. Triplet evaluation - Fine-tuned model 

In [36]:
dataset_labels = ['full_data']

train_score_mean = []
train_score_std = []

test_score_mean = []
test_score_std = []

for datalabel in dataset_labels:
    eval_folder = f'../dataset-robust/{datalabel}/evaluation/'
    train_score = []
    test_score = []

    n_data = 5 if datalabel!='temporal_division' else 1 
    
    for idx in range(n_data):
        if datalabel!='full_data_bert':
            train_result = pd.read_csv(eval_folder + f'triplet_evaluation_roberta-base-Train_idx{idx}_epoch3_results.csv')
            test_result = pd.read_csv(eval_folder + f'triplet_evaluation_roberta-base-Test_idx{idx}_epoch3_results.csv')
        
        if datalabel=='full_data_bert':
            train_result = pd.read_csv(eval_folder + f'triplet_evaluation_bert-base-Train_idx{idx}_epoch3_results.csv')
            test_result = pd.read_csv(eval_folder + f'triplet_evaluation_bert-base-Test_idx{idx}_epoch3_results.csv')

        train_score.append(train_result['accuracy_cosinus'].iloc[0])
        test_score.append(test_result['accuracy_cosinus'].iloc[0])

    train_score_mean.append(np.mean(train_score))
    test_score_mean.append(np.mean(test_score))

    train_score_std.append(np.std(train_score))
    test_score_std.append(np.std(test_score))

In [34]:
df_score = pd.DataFrame({'labels':dataset_labels, 'train_score_mean':train_score_mean, 'test_score_mean':test_score_mean, 'train_score_std':train_score_std, 'test_score_std':test_score_std} )

df_score

,labels,train_score_mean,test_score_mean,train_score_std,test_score_std
0,full_data,0.94589,0.673628,0.001307,0.002052


### 2. Triplet evaluation - base S-BERT model (before fine-tuning)

In [38]:
dataset_labels = ['full_data']

train_score_mean = []
train_score_std = []

test_score_mean = []
test_score_std = []

for datalabel in dataset_labels:
    eval_folder = f'../dataset-robust/{datalabel}/evaluation/'
    train_score = []
    test_score = []

    n_data = 5
    
    for idx in range(n_data):
        if datalabel=='full_data':
            train_result = pd.read_csv(eval_folder + f'triplet_evaluation_roberta-base-Train_idx{idx}_epoch3_before_finetune_results.csv')
            test_result = pd.read_csv(eval_folder + f'triplet_evaluation_roberta-base-Test_idx{idx}_epoch3_before_finetune_results.csv')
        
        if datalabel=='full_data_bert':
            train_result = pd.read_csv(eval_folder + f'triplet_evaluation_bert-base-Train_idx{idx}_epoch3_before_finetune_results.csv')
            test_result = pd.read_csv(eval_folder + f'triplet_evaluation_bert-base-Test_idx{idx}_epoch3_before_finetune_results.csv')

        train_score.append(train_result['accuracy_cosinus'].iloc[0])
        test_score.append(test_result['accuracy_cosinus'].iloc[0])

    train_score_mean.append(np.mean(train_score))
    test_score_mean.append(np.mean(test_score))

    train_score_std.append(np.std(train_score))
    test_score_std.append(np.std(test_score))

df_score_base = pd.DataFrame({'labels':dataset_labels, 'train_score_mean':train_score_mean, 'test_score_mean':test_score_mean, 'train_score_std':train_score_std, 'test_score_std':test_score_std} )
df_score_base

,labels,train_score_mean,test_score_mean,train_score_std,test_score_std
0,full_data,0.397342,0.37584,0.000783,0.002584


## STS-B Semantic Textual Similarity Benchmark

In [45]:
sts = datasets.load_dataset('glue', 'stsb', split='validation')
sts = sts.map(lambda x: {'label': x['label'] / 5.0})

samples = []
for sample in sts:
    samples.append(InputExample(
      texts = [sample['sentence1'], sample['sentence2']],
      label = sample['label']
    ))
    
embedding_evaluator = EmbeddingSimilarityEvaluator.from_input_examples(
    samples, write_csv=False
)

score = embedding_evaluator(model)


In [50]:
model = SentenceTransformer('../model/full_data/roberta-base_idx0_epoch3/')

print(score['spearman_euclidean'])

0.7163496029014318
